In [ ]:
#| hide
import atexit
import os
from pathlib import Path
from shutil import rmtree
from tempfile import gettempdir

from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import read_nb as _read_raw_nb
from fastcore.nbio import write_nb as _write_nb

from nbskill.convert import convert
from nbskill.execute import exec_nb
from nbskill.mcp import create_mcp
from nbskill.read import context
from nbskill.review import diff_nb, style_check
from nbskill.write import update_cell, write_nb


def _find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").exists(): return folder
    return start


project_root = _find_project_root()
workspace = None
_readme_cleanup_registered = False


def _cleanup_readme_workspace():
    if workspace is not None and workspace.exists(): rmtree(workspace)


def _readme_workspace():
    global workspace, _readme_cleanup_registered
    default = Path(gettempdir()) / "nbskill-readme-demo"
    workspace = Path(os.environ.get("NBSKILL_README_DEMO_DIR", default))
    _cleanup_readme_workspace()
    workspace.mkdir(parents=True)
    if not _readme_cleanup_registered:
        atexit.register(_cleanup_readme_workspace)
        _readme_cleanup_registered = True
    return workspace

# nbskill

`nbskill` is a small toolkit for working with nbdev notebooks as source code. It gives agents and humans notebook-aware MCP tools for reading, writing, executing, reviewing, converting, and serving notebooks.

The central idea is simple: keep the notebook as the source of truth, but give automation stable handles so it can make small, reviewable changes without touching raw JSON.

## The problem this project solves

Raw notebooks are awkward for coding agents. Cell ids, outputs, metadata, Markdown, and code are mixed together in JSON, while nbdev expects the meaningful implementation to stay in notebooks and export clean Python modules from there.

That mismatch matters in practice. A small source edit can accidentally preserve stale outputs, overwrite the wrong cell after context goes stale, or bypass the notebook story that explains why the code exists. `nbskill` gives agents a narrow set of notebook-aware tools: context readers, structured edits, execution, diffs, and diagnostics.

## How the notebooks fit together

The notebooks in `nbs/` are ordered like the toolchain itself:

1. `00_foundation.ipynb` defines the private parsing, cell, chapter, and shared helpers used everywhere else.
2. `01_read.ipynb` makes notebooks readable without JSON noise.
3. `02_write.ipynb` applies safe cell edits and exports when requested.
4. `03_execute.ipynb` executes notebooks in the local project context.
5. `04_review.ipynb` keeps review centered on code-cell diffs and style feedback.
6. `05_convert.ipynb` turns Python files into nbdev notebooks.
7. `06_skill.ipynb` installs the bundled agent skill.
8. `07_mcp.ipynb` wraps the functions as an MCP server.
9. `08_edit_interactive.ipynb` runs a bounded edit loop over one notebook.
10. `09_parallel.ipynb` provides locks so concurrent notebook operations stay orderly.
11. `10_graph.ipynb` builds a static symbol graph for definitions, callers, and private-helper reports.
12. `11_agent_workbench.ipynb` compiles taste, context, budgets, and gates into a small-diff agent workbench.

## Production readiness

Production readiness starts with contracts, then behavior, then cleanup. The repository treats notebooks as source, so every public feature needs a small executable contract before cells are split or code is moved.

The production core is intentionally narrow:

| Area | MCP tools | Public API | Status |
| --- | --- | --- | --- |
| Reading | `context` | yes | core |
| Editing | `edit_notebook` | yes | core |
| Execution and review | `exec_nb`, `diff_nb`, `style_check`, `doctor` | yes | core |
| Conversion | `convert` | yes | supporting |

### Production workflow

1. Edit the source notebook, not generated Python.
2. Keep the intended behavior covered by a contract cell or behavior-test notebook.
3. Export generated modules through notebook-aware MCP edits.
4. Run `doctor(scopes="error,warning")` on the touched notebooks or project.
5. Run focused execution checks with `exec_nb(check_only=True)`.
6. Run `style_check(changed_only=True)` for the changed notebooks.
7. Only promote strict style gates after the current warning backlog has been paid down.


## A tiny notebook to work on

The examples below create a temporary notebook and show the behavior the MCP tools expose to agents. Nothing here edits this repository.

In [ ]:
workspace = _readme_workspace()
demo_nb = workspace / "demo.ipynb"

_write_nb(new_nb([
    mk_cell("## A tiny notebook", cell_type="markdown"),
    mk_cell("x = 2\nx + 3", cell_type="code"),
]), demo_nb)

## Reading: choose the smallest useful context

The context API is one deliberate entry point. `context` accepts a project, notebook path, chapter title, cell id, or public symbol target. Cell and symbol lookups include symbol graph payloads for caller/callee impact.

The MCP tools keep printing details hidden unless the returned context is useful to the agent.

In [ ]:
context(str(demo_nb))

## Writing: add cells through MCP

Use `edit_notebook` when adding examples, tests, explanatory sections, or source changes. The MCP tool accepts structured edits and can replace cells, patch line ranges, insert cells before or after stable ids, delete or move cells, and run notebook-wide `replace_text`/`replace_texts` renames.

That lets nbskill preserve notebook structure, clear stale execution state where needed, and keep generated modules in sync when the notebook has an export target.

For larger notebook refactors, use `agent_workbench` with a tight goal, small budgets, and explicit verification. Keep the loop notebook-first: read the smallest useful context, make a focused edit, then check the changed cells before expanding the scope.

In [ ]:
write_nb(str(demo_nb), chr(10).join([
    "%%markdown",
    "## Result",
    "The next cell computes from the earlier value.",
    "---",
    "%%code",
    "answer = x * 10",
    "answer",
]))
context("Result", scope=str(demo_nb))

## Updating: use ids for precise edits

Use `edit_notebook` with `cell_id` when replacing one cell or changing a line range. Use `target="all"` with `replace_text` or `replace_texts` for notebook-level renames. Pass the `expected_hash` from the latest context read when stale context should fail instead of overwriting newer work.

Keep the edit narrow, then immediately inspect `diff_nb` or run the smallest useful `exec_nb` check.


In [ ]:
answer_cell = next(cell for cell in _read_raw_nb(demo_nb).cells if "answer = x * 10" in cell.source)
update_cell(str(demo_nb), "answer = x * 12\nanswer", cell_id=answer_cell.id)
_ = context(answer_cell.id, scope=str(demo_nb))

## Executing: run the notebook as a notebook

`exec_nb` uses `execnb` and adds the notebook directory plus the project root to the import path. That lets tests and examples behave like they do inside an nbdev project.

This matters because many notebook bugs only appear when cells are run in order with the same imports, fixtures, and local package path a real user gets. A normal Python import check can miss that story; executing the notebook checks the literate source itself.

In [ ]:
_ = exec_nb(str(demo_nb), timeout=5, show_output=True)

## Reviewing: look at behavior and code changes

`context` answers the question "what should I know before changing this implementation?" with exact source, nearby Markdown, examples/tests, and symbol graph payloads when the target is a public symbol or cell id. `diff_nb` keeps review focused on code-cell source rather than outputs and metadata.

These tools keep review at the level a maintainer cares about. `context` reconstructs the local rationale and impact around a function, while `diff_nb` filters out notebook churn so a reviewer can see whether the implementation changed.

In [ ]:
_ = context("write_nb", scope=str(project_root / "nbs/02_write.ipynb"))
_ = diff_nb(str(project_root / "nbs/02_write.ipynb"), ref_a=None)

## Converting: bootstrap nbdev notebooks from Python

`convert` parses Python with `ast`, creates nbdev notebooks, and can also migrate a pure-Python package into an nbdev project structure.

In [ ]:
sample_py = workspace / "sample_tool.py"
sample_py.write_text("def double(x):\n    return x * 2\n", encoding="utf-8")
converted_nb = workspace / "sample_tool.ipynb"

_ = convert(str(sample_py), dest=str(converted_nb))
_ = context(str(converted_nb))

## Serving the workflow through MCP

The MCP server in `07_mcp.ipynb` registers the same operations as tools. The server layer is intentionally thin: it captures stdout, holds notebook locks around file operations, and delegates the real behavior back to the notebook-defined functions.

In [ ]:
mcp = create_mcp()
type(mcp).__name__

'FastMCP'

## A realistic agent workflow

A typical agent session should be small and reversible. First check that the MCP server is connected, then use the smallest reader that answers the current question.

```python
healthcheck()
context(target="project", scope=".")
context(target="nbs/02_write.ipynb")
context(target="Updating", scope="nbs/02_write.ipynb")
context(target="write_nb", scope="nbs/02_write.ipynb")
```

Then edit the cell that actually needs to change and run focused verification.

After inspecting the precise cell, edit by stable cell id or a narrow line range. MCP notebook edits export automatically when the notebook has an export target.

```python
edit_notebook(
    path="nbs/02_write.ipynb",
    edits=[dict(op="replace_cell", cell_id="abc123", source_lines=["def target():", "    return 'updated'"])],
)
```


Finish with a review or execution tool depending on what changed. Use `diff_nb` for implementation edits and `exec_nb` when the notebook behavior needs to be checked end to end.

```python
diff_nb(path="nbs/02_write.ipynb")
exec_nb(path="nbs/02_write.ipynb", timeout=10, show_output=True)
```


<!-- nbskill-skill:start -->
# Jupyter Notebooks

Use this skill when a repository treats notebooks as source files, especially nbdev projects where `nbs/*.ipynb` exports to Python modules. Prefer the nbskill MCP server as the normal interface: inspect, edit, execute, review, and diagnose notebooks through MCP tools instead of raw `.ipynb` JSON, generated `.py` files, or ad hoc file manipulation.

## Setup

Call `healthcheck` first when MCP tools are available. It confirms the server is alive, reports capabilities, and gives reconnect hints when the client is using stale tool metadata.

## Core Workflow

1. Use `healthcheck` before notebook work or after reinstalling/exporting tool signatures.
2. Use `context` for project, notebook, chapter title, cell id, or public symbol targets. Cell and symbol targets include graph-oriented caller/callee impact.
3. Keep notebook craft in the loop: preserve the story, add rationale before code, and put examples or tests after implementation cells.
4. Use `edit_notebook` for notebook edits.
5. Verify with `exec_nb`, `diff_nb`, `style_check`, or `doctor` before handing work back.

## Notebook Craft

A good notebook has a story. Move one step at a time: describe the problem in Markdown, export the implementation, demonstrate it with a small example and visible output when useful, then protect it with a small test. Larger features should grow from earlier cells rather than appear as one big code block.

Keep cells small and semantic. A cell should usually be one of these things: Markdown rationale, imports, exported code, private implementation, a visible example, or a focused test. Avoid cells that mix several jobs, duplicate imports, hide unused code, or bundle many assertions together.

Documentation should explain the shape of the code, not just repeat it. Say why the behavior exists, what problem it solves, what tradeoff it chooses, and why an obvious alternative is not being used. For shared helpers, include cross-references: where the symbol is called, why those callers need it, and whether a private helper should be promoted before another notebook imports it.

Examples should be executable and useful to a reader. Prefer short examples close to the feature they demonstrate, with visible outputs when the output helps understanding. Tests should be small, local, and named by the single behavior they protect.

Notebook directives matter: cells exported to Python start with `#| export`, cells that should not execute during the test phase include `#| eval: false`, and cells that should not appear in documentation, such as test cells, start with `#| hide`.

## References

Open references only when the core workflow is not enough:

- `references/mcp-tools.md` for detailed MCP behavior, reconnect notes, and concurrency behavior.
- `references/mcp-tool-report.md` for MCP feature groups and tool-count reduction candidates.
- `references/conversion.md` for converting Python files, folders, or projects with `convert`.
- `references/extended-tools.md` for symbol docs, review, graph reports, and edit-interactive plans.
<!-- nbskill-skill:end -->

## Agent editing policy

Notebook edits should stay small, readable, and reviewable. Use MCP tools for normal notebook work, keep generated files in sync by exporting from the notebook source, and avoid raw notebook JSON unless you are diagnosing the tool itself.

A good edit improves both behavior and the surrounding explanation. Add or preserve rationale, cross-references, visible examples, and small tests when they help the notebook read as a coherent build-up rather than a pile of cells.

Do not hide broad changes inside oversized cells, duplicate imports across the same notebook scope, leave unused code behind, or bundle unrelated assertions into one test cell. Before stopping, inspect `diff_nb` and run a style-oriented check with `doctor(scopes="error,warning,style")` or `style_check` for the touched notebooks.